In [22]:
# Group suffixes into natural categories

import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [23]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data.parquet"))

In [24]:
sfx_df = {}
for idx, row in df.iterrows():
    title = row["title"]
    parsed_casenum = parse_casenum(title)
    suffixes = parsed_casenum['suffixes']
    for sfx in suffixes:
        if sfx not in sfx_df:
            sfx_df[sfx] = 0
        sfx_df[sfx] += 1
sfx_df = pd.DataFrame(list(sfx_df.items()), columns=['suffix', 'count'])
sfx_df = sfx_df.sort_values(by='count', ascending=False)

sfx_df.to_csv(os.path.join(DATA_PATH, "intermediate_data/cpc", "suffix_counts.csv"), index=False)

In [ ]:
"""
Manual step:
Take suffix_counts.csv, upload it along with "Active Prefix Suffix Codes.csv" to a LLM, and give the LLM this prompt:

---

Attached are `suffix_counts.csv` and `Active Prefix Suffix Codes.csv`, both of which pertain to planning and zoning in the City of Los Angeles. Please analyze these files and group the suffixes into 10-15 natural categories based on their usage and meaning. It is acceptable that a group contains only one suffix, especially for the most frequent suffixes. 

Return a csv with the same number of rows as `suffix_counts.csv` with four additional columns: `suffix_desc`, `group`, `group_count`, and `group_desc`. `suffix_desc` should provide a brief description of the suffix. `group` should be a 2 or 3 character alphanumeric group code, and simply equal to the suffix if one suffix forms a group on its own. `group_count` gives the total count of that group. `group_desc` should provide a brief description of the group. Sort the csv by `group_count` then `count` in descending order.

---

Store the resulting csv in LOCAL_PATH/intermediate_data/cpc/suffix_groups.csv
"""
''

''